# Feature Engineering cho Flight Delay Prediction

Mục tiêu: tạo các feature mới, one-hot encode các cột phân loại và xuất dataset đã xử lý sang `data/processed_flight_data.csv`.

## 1. Import và load dữ liệu
- Dùng `pandas` và `numpy` để xử lý dữ liệu.
- Load file nguồn từ `data/DelayedFlights.csv`.

In [2]:
import pandas as pd
import numpy as np

csv_path = '../data/cleaned_flights.csv'
df = pd.read_csv(csv_path, low_memory=False)
print('Original shape:', df.shape)
df.head()

Original shape: (1868454, 41)


,Unnamed: 0,Month,DayofMonth,DayOfWeek,DepTime,ArrTime,CRSElapsedTime,ArrDelay,DepDelay,Origin,...,UniqueCarrier_NW,UniqueCarrier_OH,UniqueCarrier_OO,UniqueCarrier_UA,UniqueCarrier_US,UniqueCarrier_WN,UniqueCarrier_XE,UniqueCarrier_YV,Origin_Freq,Dest_Freq
0,0,1,3,4,2003.0,2211.0,150.0,-14.0,8.0,IAD,...,0,0,0,0,0,1,0,0,0.011126,0.011779
1,1,1,3,4,754.0,1002.0,145.0,2.0,19.0,IAD,...,0,0,0,0,0,1,0,0,0.011126,0.011779
2,2,1,3,4,628.0,804.0,90.0,14.0,8.0,IND,...,0,0,0,0,0,1,0,0,0.004970,0.015411
3,4,1,3,4,1829.0,1959.0,90.0,34.0,34.0,IND,...,0,0,0,0,0,1,0,0,0.004970,0.015411
4,5,1,3,4,1940.0,2121.0,115.0,11.0,25.0,IND,...,0,0,0,0,0,1,0,0,0.004970,0.004983


## 2. Tối ưu memory usage ban đầu
- Chuyển một số cột số sang dtype nhỏ hơn.
- Chuyển cột phân loại sang `category` nếu hợp lý.

In [3]:
int_cols = [col for col in df.select_dtypes(include=['int64']).columns if col not in ['Year']]
float_cols = df.select_dtypes(include=['float64']).columns.tolist()

for col in int_cols:
    df[col] = pd.to_numeric(df[col], downcast='integer')
for col in float_cols:
    df[col] = pd.to_numeric(df[col], downcast='float')

for col in ['UniqueCarrier', 'Origin', 'Dest']:
    if col in df.columns:
        df[col] = df[col].astype('category')

print('Memory usage after downcast:', df.memory_usage(deep=True).sum() / 1024**2, 'MB')
df.info(verbose=False)

Memory usage after downcast: 169.28672122955322 MB
<class 'pandas.DataFrame'>
RangeIndex: 1868454 entries, 0 to 1868453
Columns: 41 entries, Unnamed: 0 to Dest_Freq
dtypes: category(2), float32(14), int16(2), int8(22), str(1)
memory usage: 169.3 MB


## 3. Tạo các feature mới
- `DepHour`: giờ khởi hành dự kiến.
- `IsWeekend`: xác định chuyến bay cuối tuần.
- `IsRushHour`: xác định giờ cao điểm.
- `FlightPeriod`: nhóm theo mùa/mốc thời gian trong năm.

Giải thích:
- `DepHour` thường liên quan tới mức độ delay thống kê khác nhau theo thời điểm trong ngày.
- `IsWeekend` giúp mô hình nhận ra thay đổi hành vi chuyến bay cuối tuần.
- `IsRushHour` phản ánh tắc nghẽn sân bay buổi sáng và chiều.
- `FlightPeriod` nắm bắt mùa bay và thay đổi thời tiết/hành vi theo tháng.

In [4]:
def extract_hour_from_time(value):
    if pd.isna(value):
        return np.nan
    try:
        time_int = int(value)
    except (ValueError, TypeError):
        return np.nan

    if time_int < 0:
        return np.nan
    return (time_int // 100) % 24

df = df.copy()
if 'CRSDepTime' in df.columns:
    df['DepHour'] = df['CRSDepTime'].map(extract_hour_from_time).astype('Int64')
elif 'DepTime' in df.columns:
    df['DepHour'] = df['DepTime'].map(extract_hour_from_time).astype('Int64')
else:
    df['DepHour'] = np.nan

if 'DayOfWeek' in df.columns:
    df['IsWeekend'] = df['DayOfWeek'].isin([6, 7]).astype('int8')
else:
    df['IsWeekend'] = 0

rush_hours = range(6, 10)  # 6-9 am
if 'DepHour' in df.columns:
    df['IsRushHour'] = df['DepHour'].isin(rush_hours).astype('Int64')
else:
    df['IsRushHour'] = 0

season_map = {
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Fall', 10: 'Fall', 11: 'Fall'
}
if 'Month' in df.columns:
    df['FlightPeriod'] = df['Month'].map(season_map).astype('category')
else:
    df['FlightPeriod'] = 'Unknown'

df[['DepHour', 'IsWeekend', 'IsRushHour', 'FlightPeriod']].head()

,DepHour,IsWeekend,IsRushHour,FlightPeriod
0,20,0,0,Winter
1,7,0,1,Winter
2,6,0,1,Winter
3,18,0,0,Winter
4,19,0,0,Winter


## 4. Encode phân loại theo kiểu thân thiện với bộ nhớ
- Only one-hot encode low-cardinality columns such as `UniqueCarrier`.
- Keep high-cardinality columns `Origin` và `Dest` as `category` dtypes để tránh tạo quá nhiều cột.
- Thêm `Origin_Freq` và `Dest_Freq` để ghi nhận tần suất xuất hiện normalized.

In [5]:
# Memory and shape before encoding
print('Shape before encoding:', df.shape)
print('Memory usage before encoding: {:.2f} MB'.format(df.memory_usage(deep=True).sum() / 1024**2))

low_cardinality_cols = [col for col in ['UniqueCarrier'] if col in df.columns]
high_cardinality_cols = [col for col in ['Origin', 'Dest'] if col in df.columns]

# Keep high-cardinality categorical columns as pandas category dtype to save memory.
# Avoid one-hot encoding Origin/Dest because each unique airport would create many sparse columns.
for col in high_cardinality_cols:
    df[col] = df[col].cat.as_ordered() if df[col].dtype.name == 'category' else df[col].astype('category')
    freq_col = f'{col}_Freq'
    df[freq_col] = df[col].map(df[col].value_counts(normalize=True)).astype('float32')

# One-hot encode only the low-cardinality categorical column(s).
if low_cardinality_cols:
    df = pd.get_dummies(df, columns=low_cardinality_cols, prefix=low_cardinality_cols, dtype=np.uint8)

print('Shape after encoding:', df.shape)
print('Memory usage after encoding: {:.2f} MB'.format(df.memory_usage(deep=True).sum() / 1024**2))

# Show the new frequency columns and one-hot encoded carrier features.
display_cols = high_cardinality_cols + [f'{col}_Freq' for col in high_cardinality_cols] + [col for col in df.columns if col.startswith('UniqueCarrier_')]
df[display_cols].head()

Shape before encoding: (1868454, 45)
Memory usage before encoding: 204.92 MB
Shape after encoding: (1868454, 45)
Memory usage after encoding: 204.92 MB


,Origin,Dest,Origin_Freq,Dest_Freq,UniqueCarrier_AA,UniqueCarrier_AQ,UniqueCarrier_AS,UniqueCarrier_B6,UniqueCarrier_CO,UniqueCarrier_DL,...,UniqueCarrier_HA,UniqueCarrier_MQ,UniqueCarrier_NW,UniqueCarrier_OH,UniqueCarrier_OO,UniqueCarrier_UA,UniqueCarrier_US,UniqueCarrier_WN,UniqueCarrier_XE,UniqueCarrier_YV
0,IAD,TPA,0.011126,0.011779,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,IAD,TPA,0.011126,0.011779,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,IND,BWI,0.004970,0.015411,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,IND,BWI,0.004970,0.015411,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,IND,JAX,0.004970,0.004983,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


## 5. Tối ưu memory usage sau khi feature engineering
- Kiểm tra lại bộ nhớ và downcast các cột số.
- Lưu ý: one-hot encoded columns đã là `uint8`, nên khá tiết kiệm.

In [6]:
for col in df.select_dtypes(include=['int64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='integer')
for col in df.select_dtypes(include=['float64']).columns:
    df[col] = pd.to_numeric(df[col], downcast='float')

print('Memory usage after feature engineering:', df.memory_usage(deep=True).sum() / 1024**2, 'MB')
df.info(verbose=False)

Memory usage after feature engineering: 179.97815322875977 MB
<class 'pandas.DataFrame'>
RangeIndex: 1868454 entries, 0 to 1868453
Columns: 45 entries, Unnamed: 0 to FlightPeriod
dtypes: Int8(2), category(3), float32(14), int16(2), int8(23), str(1)
memory usage: 180.0 MB


## 6. Xuất dataset đã xử lý
- Lưu file kết quả vào `data/processed_flight_data.csv`.
- Dùng `index=False` để tránh lưu cột index không cần thiết.

In [7]:
output_path = '../data/processed_flight_data.csv'
df.to_csv(output_path, index=False)
print('Saved processed dataset to', output_path)

Saved processed dataset to ../data/processed_flight_data.csv
